# Transformations: answering questions about the fire calls data

This notebook answers 10 questions about the San Francisco fire calls table `dev.spark_db.sf_fire_calls`.

Each question is answered **twice**:
1. with the **DataFrame API** (Python cell)
2. with **Spark SQL** (the `%sql` cell right below it)

Both versions produce the same plan, so they return the same result at the same speed.

### Methods used in this notebook

| DataFrame API | SQL | What it does |
|---|---|---|
| `select("col")` | `select col` | Keep some columns |
| `selectExpr("sql expression")` | `select expression` | Select using SQL expressions (`count(...)`, `year(...)`, `as` aliases) |
| `where("cond")` / `filter("cond")` | `where cond` | Keep rows that match a condition (the two methods are the same) |
| `distinct()` | `select distinct` | Remove duplicate rows |
| `groupBy("col").count()` | `group by col` + `count(*)` | Count rows per group |
| `orderBy("col", ascending=False)` | `order by col desc` | Sort |
| `show()` / `display()` | (runs automatically) | **Actions**: run the query and print the result |

> `display()` draws an interactive table only in Databricks notebooks. In VS Code, `display(df)` prints only the schema, so use `.show()` to see rows.
> In the same way, a `%sql` cell here shows only the result's schema (`DataFrame[...]`), not the rows.

## Setup: load the table

In [1]:
# Load the table as a DataFrame (only the schema is read, no data yet).
# `spark` must already exist in the kernel. After a kernel restart, add:
#   from pyspark.shell import spark
fire_df = spark.read.table("dev.spark_db.sf_fire_calls")

## Q1. How many distinct types of calls were made to the fire department?

`count(distinct ...)` counts unique values. Nulls are not counted.
The result is a single row with a single number.

In [ ]:
# count(distinct ...) counts unique call types; "as" names the result column
q1_df = (
    fire_df.selectExpr("count(distinct callType) as distinct_call_types_count")
)

# Action: run the query and print the result
q1_df.show()

+-------------------------+
|distinct_call_types_count|
+-------------------------+
|                       30|
+-------------------------+



In [6]:
%sql
-- Same query in SQL: count the unique call types
select count(distinct callType) as distinct_call_types_count
from dev.spark_db.sf_fire_calls

,distinct_call_types_count
0,30


## Q2. What were the distinct types of calls made to the fire department?

Q1 counted the call types. This lists them: drop rows with no call type, keep only that column, and remove duplicates with `distinct()`.

In [ ]:
q2_df = (fire_df.where("CallType is not null")                 # skip rows with no call type
               .selectExpr("CallType as distinct_call_type")   # keep 1 column, renamed
               .distinct()                                     # one row per call type
)

# Action: show the list (use q2_df.show(30, truncate=False) outside Databricks)
display(q2_df)

In [ ]:
%sql
-- "select distinct" removes duplicate rows, like .distinct()
select distinct CallType as distinct_call_types
from dev.spark_db.sf_fire_calls
where CallType is not null

## Q3. Find all responses where the delay was greater than 5 minutes

The `Delay` column is the response time in minutes. Filter on it and keep only the call number and the delay.

In [ ]:
(fire_df.where("Delay > 5")              # only calls with a delay over 5 minutes
       .select("CallNumber", "Delay")   # keep 2 columns
       .display()                       # action: show the result
)

In [ ]:
%sql
-- Calls with a response delay over 5 minutes
select CallNumber, Delay
from dev.spark_db.sf_fire_calls
where Delay > 5

## Q4. What were the most common call types?

Group by call type, count the calls in each group, and sort from most to least.
`groupBy(...).count()` adds a column named `count`.

In [ ]:
(fire_df.select("CallType")
    .where("CallType is not null")       # skip rows with no call type
    .groupBy("CallType")                 # one group per call type
    .count()                             # number of calls in each group -> "count" column
    .orderBy("count", ascending=False)   # most common first
    .show()                              # action
)

In [ ]:
%sql
-- count(*) per group, most common first
select CallType, count(*) as count
from dev.spark_db.sf_fire_calls
where CallType is not null
group by CallType
order by count desc

## Q5. What zip codes accounted for the most common calls?

Same as Q4, but grouped by **call type and zip code** together, so each row is one (call type, zip code) pair.

> Column names are not case-sensitive in Spark, so `ZipCode` and `Zipcode` refer to the same column.

In [ ]:
(fire_df.select("CallType", "ZipCode")
    .where("CallType is not null")       # skip rows with no call type
    .groupBy("CallType", "Zipcode")      # one group per (call type, zip code) pair
    .count()                             # calls in each pair
    .orderBy("count", ascending=False)   # biggest first
    .display()                           # action
)

In [ ]:
%sql
-- Calls per (call type, zip code) pair, biggest first
select CallType, ZipCode, count(*) as count
from dev.spark_db.sf_fire_calls
where CallType is not null
group by CallType, Zipcode
order by count desc

## Q6. What San Francisco neighborhoods are in the zip codes 94102 and 94103?

Filter on the 2 zip codes, then use `distinct()` so each (neighborhood, zip code) pair appears once.

> `Zipcode` is a string column. Spark converts it automatically when compared with the number `94102`. Writing `Zipcode in ('94102', '94103')` does the same thing and is shorter.

In [ ]:
(fire_df.select("Neighborhood", "Zipcode")
    .where("Zipcode== 94102 or Zipcode == 94103")   # only these 2 zip codes
    .distinct()                                     # each pair once
    .display()                                      # action
)

In [ ]:
%sql
-- Unique (neighborhood, zip code) pairs in the 2 zip codes
select distinct Neighborhood, Zipcode
from dev.spark_db.sf_fire_calls
where Zipcode== 94102 or Zipcode == 94103

## Q7. What were the total number of alarms, and the average, min and max response times?

Aggregate functions over the **whole table** (no `groupBy`), so the result is a single row.

> Notes:
> - `sum(NumAlarms)` adds up the **alarms**, not the calls. The total number of calls would be `count(*)`.
> - Without an alias, the columns are named after the expressions (`sum(NumAlarms)`, `avg(Delay)`, ...). Add `as total_alarms` etc. for cleaner names.

In [ ]:
# Several aggregates at once over the whole table -> one row
(fire_df.selectExpr("sum(NumAlarms)", "avg(Delay)", "min(Delay)", "max(Delay)")
        .show()   # action
)

In [ ]:
%sql
-- Totals and response-time stats over the whole table
select sum(NumAlarms), avg(Delay), min(Delay), max(Delay)
from dev.spark_db.sf_fire_calls

## Q8. How many distinct years of data are in the table?

`year(CallDate)` takes the year out of a date. `distinct()` keeps each year once, and `orderBy` sorts them.
This lists the years; add `.count()` to get just the number.

In [ ]:
(fire_df.selectExpr("year(CallDate) as year_num")   # year part of the call date
    .distinct()                                     # each year once
    .orderBy("year_num")                            # oldest first
    .show()                                         # action
)

In [ ]:
%sql
-- Unique years in the data, oldest first
select distinct year(CallDate) as year_num
from dev.spark_db.sf_fire_calls
order by year_num

## Q9. What week of the year in 2018 had the most fire calls?

`weekofyear(CallDate)` gives the week number (1 to 53). Keep only 2018, count the calls per week, and sort.
The first row is the busiest week.

In [ ]:
(fire_df.selectExpr("weekofyear(CallDate) as week_year")   # week number of each call
    .where("year(CallDate) == 2018")                        # only calls from 2018
    .groupBy('week_year')                                   # one group per week
    .count()                                                # calls per week
    .orderBy('count', ascending=False)                      # busiest week first
    .display()                                              # action
)

In [ ]:
%sql
-- Calls per week in 2018, busiest week first
select weekofyear(CallDate) as week_year, count(*) as count
from dev.spark_db.sf_fire_calls
where year(CallDate) == 2018
group by week_year
order by count desc

## Q10. What neighborhoods in San Francisco had the worst response times in 2018?

This version lists the delay of **every** 2018 call with its neighborhood.

> To rank neighborhoods, you would group and average:
> ```python
> (fire_df.where("year(CallDate) == 2018")
>     .groupBy("Neighborhood")
>     .agg(F.avg("Delay").alias("avg_delay"))      # needs: from pyspark.sql import functions as F
>     .orderBy("avg_delay", ascending=False)
>     .show())
> ```

In [ ]:
(fire_df.select("Neighborhood", "Delay")
    .filter("year(CallDate) == 2018")   # filter() is the same as where(); only 2018 calls
    .show()                             # action
)

In [ ]:
%sql
-- Delay of each 2018 call, with its neighborhood
select Neighborhood, Delay
from dev.spark_db.sf_fire_calls
where year(CallDate) == 2018

## Key takeaways

- Every question here is a chain of **transformations** (`select`, `where`, `groupBy`, `orderBy`, `distinct`) ending in one **action** (`show` or `display`).
- `where()` and `filter()` are the same method.
- `selectExpr()` lets you use SQL expressions such as `count(distinct ...)`, `year(...)` and `as` aliases inside DataFrame code.
- `groupBy(...).count()` creates a column named `count`, which you can then sort by.
- The DataFrame and SQL versions build the same plan, so pick the one that reads best.